# PCA 2D Comparison — DenseNet121 / MedMamba / Swin Transformer

Reads `*_embeddings.npz` files produced by `extract_embeddings.py` and renders a 1x3 PCA comparison figure.

Run `extract_embeddings.py` first to populate `./embeddings/all_models_test_set_128dim/`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA

EMBEDDINGS_DIR = Path.cwd() / "embeddings" / "all_models_test_set_128dim"
OUTPUT_PDF = Path.cwd() / "pca_comparison_3models.pdf"

models_config = [
    ("DenseNet121", EMBEDDINGS_DIR / "densenet121_embeddings.npz"),
    ("MedMamba", EMBEDDINGS_DIR / "medmamba_embeddings.npz"),
    ("Swin Transformer", EMBEDDINGS_DIR / "swin_transformer_embeddings.npz"),
]

class_names = {0: "Public (Medicaid/Medicare)", 1: "Private"}
colors = {0: "tab:blue", 1: "tab:orange"}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for i, (model_name, npz_path) in enumerate(models_config):
    ax = axes[i]
    print(f"Processing {model_name}...")

    if not os.path.exists(npz_path):
        raise FileNotFoundError(
            f"Missing embeddings for {model_name}: {npz_path}. Run extract_embeddings.py first."
        )

    data = np.load(npz_path)
    embeddings = data["embeddings"]
    labels = data["labels"]
    if labels.ndim > 1 and labels.shape[1] > 1:
        labels = np.argmax(labels, axis=1)
    labels = labels.squeeze()

    pca = PCA(n_components=2)
    emb_pca = pca.fit_transform(embeddings)
    classes = [0, 1]

    for cls in classes:
        idx = np.where(labels == cls)[0]
        if len(idx) > 1:
            sns.kdeplot(
                x=emb_pca[idx, 0],
                y=emb_pca[idx, 1],
                fill=True,
                color=colors[cls],
                alpha=0.2,
                levels=8,
                thresh=0.05,
                ax=ax,
                zorder=1,
            )

    for cls in classes:
        idx = np.where(labels == cls)[0]
        ax.scatter(
            emb_pca[idx, 0],
            emb_pca[idx, 1],
            s=15,
            alpha=0.8,
            color=colors[cls],
            edgecolor="white",
            linewidth=0.3,
            label=class_names[cls],
            zorder=2,
        )
    ax.legend(title="Insurance Type", loc="upper right", fontsize="small")

    x_min, x_max = np.percentile(emb_pca[:, 0], [0.03, 99.9])
    y_min, y_max = np.percentile(emb_pca[:, 1], [0.03, 99.9])
    x_margin = (x_max - x_min) * 0.1
    y_margin = (y_max - y_min) * 0.1
    ax.set_xlim(x_min - x_margin, x_max + x_margin)
    ax.set_ylim(y_min - y_margin, y_max + y_margin)

    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_title(f"{model_name}\nPCA Projection", fontweight="bold", fontsize=14)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")

plt.tight_layout()
plt.savefig(OUTPUT_PDF, format="pdf")
print(f"Comparison plot saved to {OUTPUT_PDF}")
plt.show()